# SCMP Catchments: Elevation, Stream Order and Population Export

Exports three datasets for each catchment in `projects/ee-samuelaojih/assets/SCMP_SHAPEFILES` to Google Drive, clipped to the catchment boundary:

| Layer | Dataset | Native resolution | Output |
|---|---|---|---|
| Elevation | SRTM GL1 v3 (`USGS/SRTMGL1_003`) | 1 arc-second (~30 m) | GeoTIFF (Int16, metres) |
| Stream order | HydroRIVERS via WWF Free Flowing Rivers (`WWF/HydroSHEDS/v1/FreeFlowingRivers`), field `ORD_STRA` = Strahler order | vector (15 arc-second network) | Shapefile + GeoTIFF rasterised on the SRTM grid |
| Population | WorldPop Global Project 100 m (`WorldPop/GP/100m/pop`) | 3 arc-second (~100 m) | GeoTIFF (Float32, persons per pixel) |

It also exports one CSV with per-catchment statistics (elevation min/max/mean/std, population total and density, max Strahler order and stream length per order).

In [ ]:
import re
import ee

ee.Authenticate()
ee.Initialize(project='ee-samuelaojih')

## Configuration

In [ ]:
ASSET_ID = 'projects/ee-samuelaojih/assets/SCMP_SHAPEFILES'
NAME_FIELD = 'NAME'

CATCHMENTS = [
    'Wase-Taraba',
    'Misau-K Gana',
    'Gaji-Lamurde',
    'Ngada',
    'Hawul-Kilange',
    'Yedseram',
    'Zungur-Gongola',
    'Sarkin-Powa',
    'Malenda',
    "Hadejia-Jama'are",
]

DRIVE_FOLDER = 'SCMP_Catchment_Exports'
POP_YEAR = 2020        # WorldPop GP 100 m covers 2000-2020

# None = export on each dataset's native grid (EPSG:4326, no resampling).
# Set e.g. 'EPSG:32632' (UTM 32N) to reproject; EXPORT_SCALE is then used for DEM/stream rasters.
EXPORT_CRS = None
EXPORT_SCALE = 30
POP_SCALE = 100

## Load catchments and datasets

In [ ]:
catchments = ee.FeatureCollection(ASSET_ID).filter(ee.Filter.inList(NAME_FIELD, CATCHMENTS))

found = catchments.aggregate_array(NAME_FIELD).getInfo()
missing = sorted(set(CATCHMENTS) - set(found))
print(f'{len(found)} catchment features found:', found)
if missing:
    print('WARNING - not found in asset (check spelling):', missing)

# SRTM 30 m
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')

# HydroRIVERS network with Strahler order (ORD_STRA)
rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers')
RIVER_FIELDS = ['REACH_ID', 'ORD_STRA', 'ORD_CLAS', 'ORD_FLOW', 'RIV_ORD', 'DIS_AV_CMS', 'UPLAND_SKM']

# WorldPop: mosaic all countries for the year, since some catchments
# (e.g. Yedseram, Ngada) extend beyond Nigeria into Cameroon/Niger/Chad.
pop_col = ee.ImageCollection('WorldPop/GP/100m/pop').filter(ee.Filter.eq('year', POP_YEAR))
pop_proj = pop_col.filter(ee.Filter.eq('country', 'NGA')).first().projection()
pop = pop_col.select('population').mosaic().setDefaultProjection(pop_proj).rename('population')

## Helpers

In [ ]:
def safe_name(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')


def grid_kwargs(proj, scale):
    """Native grid (crs + crsTransform) unless EXPORT_CRS is set."""
    if EXPORT_CRS:
        return {'crs': EXPORT_CRS, 'scale': scale}
    info = proj.getInfo()
    return {'crs': info['crs'], 'crsTransform': info['transform']}


DEM_GRID = grid_kwargs(dem.projection(), EXPORT_SCALE)
POP_GRID = grid_kwargs(pop_proj, POP_SCALE)


def clip_rivers(geom):
    """River reaches cut to the catchment, with the clipped length in km."""
    def _clip(f):
        f = ee.Feature(f).select(RIVER_FIELDS).intersection(geom, ee.ErrorMargin(1))
        return f.set('len_km', f.length(1).divide(1000))
    return rivers.filterBounds(geom).map(_clip)


def stream_order_image(clipped, geom):
    """Rasterise Strahler order (0 = no mapped stream)."""
    return (clipped.reduceToImage(['ORD_STRA'], ee.Reducer.max())
            .unmask(0).clip(geom).toByte().rename('strahler_order'))


ORDERS = list(range(1, 11))


def summarize(feature):
    geom = feature.geometry()
    area_km2 = geom.area(1).divide(1e6)

    elev = dem.reduceRegion(
        reducer=ee.Reducer.minMax()
            .combine(ee.Reducer.mean(), sharedInputs=True)
            .combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=geom, crs=dem.projection(), maxPixels=1e13, tileScale=4)

    pop_sum = ee.Number(pop.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, crs=pop_proj,
        maxPixels=1e13, tileScale=4).get('population'))

    clipped = clip_rivers(geom)
    stats = {
        'NAME': feature.get(NAME_FIELD),
        'area_km2': area_km2,
        'elev_min_m': elev.get('elevation_min'),
        'elev_max_m': elev.get('elevation_max'),
        'elev_mean_m': elev.get('elevation_mean'),
        'elev_std_m': elev.get('elevation_stdDev'),
        f'pop_{POP_YEAR}': pop_sum,
        'pop_density_km2': pop_sum.divide(area_km2),
        'popsum_asset': feature.get('_popsum'),
        'max_strahler_order': clipped.aggregate_max('ORD_STRA'),
        'stream_km_total': clipped.aggregate_sum('len_km'),
    }
    for o in ORDERS:
        stats[f'stream_km_o{o}'] = clipped.filter(ee.Filter.eq('ORD_STRA', o)).aggregate_sum('len_km')
    return ee.Feature(None, stats)

## Start exports (per catchment rasters/vectors + summary CSV)

In [ ]:
tasks = []


def start(task):
    task.start()
    tasks.append(task)


for name in found:
    geom = catchments.filter(ee.Filter.eq(NAME_FIELD, name)).geometry()
    region = geom.bounds()
    tag = safe_name(name)

    # 1. Elevation (SRTM 30 m)
    start(ee.batch.Export.image.toDrive(
        image=dem.clip(geom).toInt16(),
        description=f'{tag}_SRTM30m_DEM',
        folder=DRIVE_FOLDER, fileNamePrefix=f'{tag}_SRTM30m_DEM',
        region=region, maxPixels=1e13, **DEM_GRID))

    # 2a. Stream order - vector
    clipped = clip_rivers(geom)
    start(ee.batch.Export.table.toDrive(
        collection=clipped,
        description=f'{tag}_StreamOrder_vector',
        folder=DRIVE_FOLDER, fileNamePrefix=f'{tag}_StreamOrder',
        fileFormat='SHP'))

    # 2b. Stream order - raster on the SRTM grid
    start(ee.batch.Export.image.toDrive(
        image=stream_order_image(clipped, geom),
        description=f'{tag}_StreamOrder_raster',
        folder=DRIVE_FOLDER, fileNamePrefix=f'{tag}_StreamOrder_30m',
        region=region, maxPixels=1e13, **DEM_GRID))

    # 3. Population (WorldPop 100 m)
    start(ee.batch.Export.image.toDrive(
        image=pop.clip(geom).toFloat(),
        description=f'{tag}_WorldPop{POP_YEAR}_100m',
        folder=DRIVE_FOLDER, fileNamePrefix=f'{tag}_WorldPop{POP_YEAR}_100m',
        region=region, maxPixels=1e13, **POP_GRID))

    print(f'Started 4 exports for {name}')

# 4. Summary statistics table
summary = catchments.map(summarize)
selectors = (['NAME', 'area_km2', 'elev_min_m', 'elev_max_m', 'elev_mean_m', 'elev_std_m',
              f'pop_{POP_YEAR}', 'pop_density_km2', 'popsum_asset',
              'max_strahler_order', 'stream_km_total']
             + [f'stream_km_o{o}' for o in ORDERS])
start(ee.batch.Export.table.toDrive(
    collection=summary,
    description='SCMP_Catchment_Summary',
    folder=DRIVE_FOLDER, fileNamePrefix='SCMP_Catchment_Summary',
    fileFormat='CSV', selectors=selectors))

print(f'\n{len(tasks)} export tasks submitted to Drive folder "{DRIVE_FOLDER}".')

## Monitor tasks
Re-run this cell to refresh. Large catchments (e.g. Hadejia-Jama'are) may be split into several GeoTIFF tiles by Earth Engine.

In [ ]:
for t in tasks:
    s = t.status()
    print(f"{s['description']:<45} {s['state']:<10} {s.get('error_message', '')}")

## Optional: preview on a map

In [ ]:
import geemap

m = geemap.Map(basemap='SATELLITE')
m.centerObject(catchments, 6)
m.addLayer(dem.clip(catchments), {'min': 150, 'max': 1500, 'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}, 'SRTM DEM')
m.addLayer(pop.clip(catchments), {'min': 0, 'max': 50, 'palette': ['ffffcc', 'fd8d3c', '800026']}, f'WorldPop {POP_YEAR}', False)
m.addLayer(rivers.filterBounds(catchments).filter(ee.Filter.gte('ORD_STRA', 3)).style(color='0066ff', width=1), {}, 'Streams (order >= 3)')
m.addLayer(catchments.style(color='ff0000', fillColor='00000000', width=2), {}, 'Catchments')
m